# 06 — Main Factorial: Mechanisms × Geometry

**Research question.** Holding budget and intercept fixed at the pilot-locked values, how do the three tie-forming mechanisms (homophily, triadic closure, popularity) interact with geometry (torus-5d, torus-2d, Poincaré) to produce structural holes?

**Design.** 3³ mechanism grid × 3 geometries × 3 replicates = **243 runs**, all at the pilot-locked settings: `budget=20`, `intercept=-5`, `n_steps=50`, snapshots at t ∈ {0, 20, 50}, `sim_seed = rep * 1000 + cell_idx`.

**Blocking.** Three replicates per (mechanism cell, geometry). Each (geometry, replicate) gets its own init with `.normalized(method="mean")` applied so mechanism coefficients are on comparable scales across geometries. Within a (geometry, replicate) block, all 27 mechanism cells share the same init → paired comparisons.

**Primary DV.** `p10_constraint` (brokerage tail — the 10th percentile of the constraint distribution). Secondary: `mean_constraint`, `c_density`, `c_hierarchy`. We do not report `c_size`; the pilot showed it is mechanically saturated at `1/budget`.

**Pilot reference.** Spec: `docs/superpowers/specs/2026-04-17-pilot-calibration-design.md`. Notebook: `05_pilot_calibration.ipynb`. See the pilot's Decisions cell for all settings.


In [ ]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from abm_core import init_torus_uniform, init_hyperbolic_uniform
from experiment_grid_search import run_grid_cell

# ---------- Pilot-locked settings ----------
N = 300
BUDGET = 20
INTERCEPT = -5.0
N_STEPS = 50
SNAPSHOT_TIMES = [0, 20, 50]
REPLICATES = 3

LEVELS = {
    "b_homophily":  [0.0, 2.0, 4.0],
    "b_triadic":    [0.0, 2.0, 4.0],
    "b_popularity": [0.0, 0.5, 1.0],
}

GEOMETRIES = ["torus5d", "torus2d", "poincare"]

OUT_DIR = Path("simulations/main_factorial")
RUNS_DIR = OUT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUT_DIR / "summary.parquet"


## Build inits per (geometry, replicate) block

All inits are normalized by mean distance so the same coefficient means the same thing across geometries. 9 inits total (3 geometries × 3 replicates).


In [ ]:
def make_init(geometry: str, seed: int):
    rng = np.random.default_rng(seed)
    if geometry == "torus5d":
        init = init_torus_uniform(n=N, d=5, rng=rng)
    elif geometry == "torus2d":
        init = init_torus_uniform(n=N, d=2, rng=rng)
    elif geometry == "poincare":
        init = init_hyperbolic_uniform(n=N, spread=1.0, rng=rng)
    else:
        raise ValueError(geometry)
    return init.normalized(method="mean")

inits = {
    (geo, rep): make_init(geo, seed=rep)
    for geo in GEOMETRIES
    for rep in range(REPLICATES)
}
for (geo, rep), init in inits.items():
    D = init.distance_matrix
    upper = D[np.triu_indices(N, k=1)]
    print(f"  {geo} rep{rep}: N={init.n}, mean_d={upper.mean():.3f}, max_d={upper.max():.3f}")


## Run the grid

243 runs in parallel via joblib. Reruns skip existing output files to enable iteration without recomputing.


In [ ]:
cells = list(product(LEVELS["b_homophily"],
                     LEVELS["b_triadic"],
                     LEVELS["b_popularity"]))
print(f"{len(cells)} mechanism cells x {len(GEOMETRIES)} geometries x {REPLICATES} reps = "
      f"{len(cells) * len(GEOMETRIES) * REPLICATES} runs")

def run_one(geo: str, rep: int, cell_idx: int, b_h: float, b_t: float, b_p: float):
    out_path = RUNS_DIR / f"{geo}_cell{cell_idx:02d}_rep{rep}.npz"
    sim_seed = rep * 1000 + cell_idx
    rows = run_grid_cell(
        init_result=inits[(geo, rep)],
        b_homophily=b_h,
        b_triadic=b_t,
        b_popularity=b_p,
        budget=BUDGET,
        n_steps=N_STEPS,
        snapshot_times=SNAPSHOT_TIMES,
        sim_seed=sim_seed,
        out_path=out_path,
        intercept=INTERCEPT,
    )
    for r in rows:
        r.update(geometry=geo, replicate=rep, cell_id=cell_idx)
    return rows

jobs = [(geo, rep, ci, b_h, b_t, b_p)
        for geo in GEOMETRIES
        for rep in range(REPLICATES)
        for ci, (b_h, b_t, b_p) in enumerate(cells)]

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_one)(*j) for j in tqdm(jobs, desc="main factorial")
)

all_rows = [r for group in results for r in group]
summary = pd.DataFrame(all_rows)
summary.to_parquet(SUMMARY_PATH)
print(f"Saved {len(summary)} rows -> {SUMMARY_PATH}")
summary.head()


## Sanity checks

- Baseline (all mechanisms = 0) should have the *lowest* `mean_constraint` across mechanism cells within each geometry (random-ish ties → less clustering → more brokers possible).
- All-high cell should have the *highest* `mean_constraint`.
- Order should hold across all three geometries.
- Replicate std at each (geometry, cell, t) should be small (≤ a few percent of the mean).


In [ ]:
summary = pd.read_parquet(SUMMARY_PATH)
end = summary[summary["t"] == N_STEPS].copy()

def label_cell(row):
    if row["b_homophily"] == 0 and row["b_triadic"] == 0 and row["b_popularity"] == 0:
        return "baseline"
    if (row["b_homophily"] == LEVELS["b_homophily"][-1]
        and row["b_triadic"] == LEVELS["b_triadic"][-1]
        and row["b_popularity"] == LEVELS["b_popularity"][-1]):
        return "all_high"
    return None

end["cell_label"] = end.apply(label_cell, axis=1)
corners = end[end["cell_label"].notna()]

print("Baseline vs all-high by geometry:")
pivot = (corners
         .groupby(["geometry", "cell_label"])
         .agg(mean_C=("mean_constraint", "mean"),
              p10_C=("p10_constraint", "mean"),
              reps=("replicate", "count"))
         .round(4))
print(pivot)
print()

print("Replicate spread (std across reps at each geometry x cell):")
spread = (end
          .groupby(["geometry", "cell_id"])
          .agg(mean_C_std=("mean_constraint", "std"),
               p10_C_std=("p10_constraint", "std")))
print(f"  max mean_C std across cells: {spread['mean_C_std'].max():.4f}")
print(f"  max p10_C std across cells:  {spread['p10_C_std'].max():.4f}")


## Main effects per geometry

For each geometry and each mechanism, plot `p10_constraint` vs that mechanism's level, marginalized across the other two mechanisms. Three panels per geometry.


In [ ]:
fig, axes = plt.subplots(len(GEOMETRIES), 3, figsize=(15, 4 * len(GEOMETRIES)), sharey="row")
for gi, geo in enumerate(GEOMETRIES):
    sub = end[end["geometry"] == geo]
    for mi, mech in enumerate(("b_homophily", "b_triadic", "b_popularity")):
        ax = axes[gi, mi]
        marg = (sub.groupby(mech)
                   .agg(mean=("p10_constraint", "mean"),
                        std=("p10_constraint", "std"))
                   .reset_index())
        ax.errorbar(marg[mech], marg["mean"], yerr=marg["std"], marker="o", capsize=3,
                    label="p10 (primary)")
        marg_mean = sub.groupby(mech)["mean_constraint"].mean().reset_index()
        ax.plot(marg_mean[mech], marg_mean["mean_constraint"], marker="s", alpha=0.6,
                label="mean")
        ax.set_xlabel(mech)
        if mi == 0:
            ax.set_ylabel(f"{geo}\nconstraint")
        ax.set_title(f"{geo} / {mech}")
        ax.legend(fontsize=8)
plt.suptitle("Main effects: constraint vs each mechanism, marginalized")
plt.tight_layout()
plt.show()


## Cross-geometry comparison

For each mechanism, overlay the three geometry curves to see whether normalization makes them line up — the Stage F prediction from the pilot. If curves align, LEVELS transfer and geometry is a minor factor. If they diverge, geometry-mechanism interaction is a real effect.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for mi, mech in enumerate(("b_homophily", "b_triadic", "b_popularity")):
    ax = axes[mi]
    for geo in GEOMETRIES:
        sub = end[end["geometry"] == geo]
        marg = sub.groupby(mech)["p10_constraint"].agg(["mean", "std"]).reset_index()
        ax.errorbar(marg[mech], marg["mean"], yerr=marg["std"], marker="o", capsize=3, label=geo)
    ax.set_xlabel(mech)
    ax.set_ylabel("p10_constraint")
    ax.set_title(f"Cross-geometry: {mech}")
    ax.legend()
plt.suptitle("Do normalized geometries give the same mechanism curves? (Stage F prediction check)")
plt.tight_layout()
plt.show()


## Two-way interaction heatmaps (per geometry)

For each geometry, 2-way heatmaps of `mean_p10_constraint` collapsing over the third mechanism. Non-additive patterns = interactions.


In [ ]:
fig, axes = plt.subplots(len(GEOMETRIES), 3, figsize=(15, 4 * len(GEOMETRIES)))
mech_pairs = [("b_homophily", "b_triadic"),
              ("b_homophily", "b_popularity"),
              ("b_triadic", "b_popularity")]

for gi, geo in enumerate(GEOMETRIES):
    sub = end[end["geometry"] == geo]
    for pi, (mx, my) in enumerate(mech_pairs):
        ax = axes[gi, pi]
        grid = (sub.groupby([mx, my])["p10_constraint"]
                   .mean()
                   .unstack(my))
        im = ax.imshow(grid.values, origin="lower", aspect="auto", cmap="viridis")
        ax.set_xticks(range(len(grid.columns)))
        ax.set_xticklabels([f"{v:g}" for v in grid.columns])
        ax.set_yticks(range(len(grid.index)))
        ax.set_yticklabels([f"{v:g}" for v in grid.index])
        ax.set_xlabel(my)
        ax.set_ylabel(f"{geo}\n{mx}" if pi == 0 else mx)
        ax.set_title(f"{geo}: p10 vs {mx} x {my}")
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                ax.text(j, i, f"{grid.values[i, j]:.3f}",
                        ha="center", va="center", color="w", fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.04)
plt.suptitle("Two-way interactions: p10_constraint surface")
plt.tight_layout()
plt.show()


## Per-node constraint distributions

For three representative cells (baseline / all-mid / all-high) across three geometries, show the per-node constraint histogram at the final snapshot. Brokerage = the left tail.


In [ ]:
REP_CELLS = {
    "baseline":  (0.0, 0.0, 0.0),
    "all_mid":   (LEVELS["b_homophily"][1], LEVELS["b_triadic"][1], LEVELS["b_popularity"][1]),
    "all_high":  (LEVELS["b_homophily"][2], LEVELS["b_triadic"][2], LEVELS["b_popularity"][2]),
}

def load_constraint(geo, cell_id, rep):
    arr = np.load(RUNS_DIR / f"{geo}_cell{cell_id:02d}_rep{rep}.npz")
    # Final snapshot = last entry in the times array
    return arr["constraint"][-1]

# Map each REP_CELLS entry to its cell_id from the factorial order
cell_to_idx = {c: i for i, c in enumerate(cells)}

fig, axes = plt.subplots(len(REP_CELLS), len(GEOMETRIES), figsize=(15, 4 * len(REP_CELLS)),
                         sharex=True, sharey=True)
for ri, (label, coefs) in enumerate(REP_CELLS.items()):
    cell_id = cell_to_idx[coefs]
    for gi, geo in enumerate(GEOMETRIES):
        ax = axes[ri, gi]
        pooled = np.concatenate([load_constraint(geo, cell_id, r) for r in range(REPLICATES)])
        ax.hist(pooled, bins=40, alpha=0.8)
        ax.axvline(0.1, color="red", linestyle="--", alpha=0.5, label="C=0.1 threshold")
        ax.set_title(f"{geo} / {label}\n({coefs[0]:g}, {coefs[1]:g}, {coefs[2]:g})")
        ax.set_xlabel("constraint")
        if gi == 0:
            ax.set_ylabel("count")
        frac_broker = (pooled < 0.1).mean()
        ax.text(0.97, 0.97, f"frac<0.1 = {frac_broker:.2f}",
                ha="right", va="top", transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.suptitle("Per-node constraint distributions: brokerage tail by geometry x cell")
plt.tight_layout()
plt.show()


## Observations

*Fill this cell after inspecting the plots above.*

### Sanity
- [ ] Baseline has the lowest mean_C per geometry?
- [ ] All-high has the highest mean_C per geometry?
- [ ] Replicate std < 5% of cell means?

### Main effects (per geometry)
- `b_homophily`: …
- `b_triadic`: …
- `b_popularity`: …

### Cross-geometry
- Do normalized curves align (Stage F prediction)? YES / NO / partial for which mechanism?
- If they diverge: which geometry is the outlier and why?

### Interactions
- Strongest 2-way interaction: …
- Does a mechanism only matter when another is off / on?

### Brokerage tail (distributions)
- Which geometry produces the largest left tail at baseline?
- Which cells produce bimodal distributions (two populations: brokers + embedded)?
- Fraction of brokers (`constraint < 0.1`) at (baseline, all-mid, all-high) for each geometry: …

### Next steps
- …
